# CKIN2 - Stickers, charms et agents

Analyse en lecture seule des inventaires et prix normalises depuis les exports locaux.
Les prix sont des observations historiques, pas des cotations actuelles. Les dates estimees sont identifiees explicitement.
Les quantites sont celles de l'export : elles ne constituent pas des snapshots historiques.


## 1. Chargement et controle des donnees

Generer les CSV depuis la racine avec `python src/process_categories.py`.


In [ ]:
from pathlib import Path
import os
import sys
import pandas as pd
import plotly.express as px
from IPython.display import display

PROJECT_ROOT = Path.cwd() if (Path.cwd() / "src").is_dir() else Path.cwd().parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))
DATA_ROOT = Path(os.environ.get("CKIN2_DATA_ROOT", str(PROJECT_ROOT)))
AS_OF = os.environ.get("CKIN2_AS_OF") or pd.Timestamp.now()
from notebook_portfolio import portfolio_report, category_reports
report = portfolio_report(DATA_ROOT, category="all", now=AS_OF)
display(report["context"])
results = category_reports(DATA_ROOT, report)
results = [r for r in results if r["category"] in ("sticker", "charm", "agent")]
for result in results:
    print(result["category"])
    display(result["summary"])
    display(result["holdings"])
    display(result["coverage"])


## 2. Evolution des prix de marche et acquisitions

Les croix indiquent les valeurs de reference aux dates d acquisition ; elles ne sont pas reliees aux courbes de marche.
La base 100, la couverture et la valorisation utilisent uniquement les releves de marche (source_column > 0).
Les dates estimees sont indiquees au survol des points. Les acquisitions figurent aussi dans une chronologie distincte.


In [ ]:
from analyze_categories import category_price_figures
for result in results:
    if not result["prices"].empty or not result["acquisitions"].empty:
        for figure in category_price_figures(result["prices"], result["acquisitions"]):
            figure.show()
    display(result["acquisitions"])
print("Références d’acquisition séparées : ni prix de marché ni coûts d’achat confirmés.")


## 3. Valeur de l'inventaire aux derniers prix connus

Le sous-total connu exclut les quantites ou prix manquants. Le total reste indisponible si la couverture est incomplete.
Les prix d'achat sans date restent utilisables pour comparer le cout et la valeur ; aucun rendement annualise n'est calcule.


In [ ]:
for result in results:
    priced = result["holdings"].loc[lambda f: f.market_value.notna() & f.quantity.gt(0)]
    if not priced.empty:
        px.bar(priced, x="item_name", y="market_value", title="Contribution au sous-total connu (EUR)").show()


## 4. Prix d'achat et limites

L'ecart indique est une difference brute, sans frais de vente. Les dates d'achat sont inconnues.
Les dates de marche intermediaires sont estimees entre les dates voisines, puis ramenees au jour civil.
Une courbe de valeur historique du portefeuille exige des snapshots de quantites : appliquer les quantites actuelles au passe serait trompeur.


In [ ]:
print("Les coûts déclarés et les résultats FIFO sont consultables dans les fiches objets du dashboard. Aucune référence d’acquisition n’est assimilée à un coût confirmé.")
